In [6]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
import matplotlib.pyplot as plt
import seaborn as sns
import itertools

pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
sns.set_style('whitegrid')

df = pd.read_csv('/content/drive/MyDrive/VOIS Internship/seasonal_agriculture_performance_dataset.csv')
seasons = ['Kharif', 'Rabi', 'Zaid']
print(df.shape)

(4000, 28)


In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
yield_groups = [df.loc[df.Season == s, 'Yield_Tonnes_Ha'].dropna() for s in seasons]

print('Normality (Shapiro-Wilk, sample of <=500) and skewness:')
for s, g in zip(seasons, yield_groups):
    sh = stats.shapiro(g.sample(min(len(g), 500), random_state=1))
    print(f'  {s}: n={len(g)}, skew={stats.skew(g):.2f}, Shapiro p={sh.pvalue:.3g}')

lev = stats.levene(*yield_groups)
print(f'\nLevene test for equal variances: stat={lev.statistic:.3f}, p={lev.pvalue:.3g}')

Normality (Shapiro-Wilk, sample of <=500) and skewness:
  Kharif: n=1772, skew=4.03, Shapiro p=2.94e-38
  Rabi: n=1607, skew=4.04, Shapiro p=6.5e-39
  Zaid: n=589, skew=3.54, Shapiro p=9.25e-38

Levene test for equal variances: stat=0.759, p=0.468


In [9]:
kw = stats.kruskal(*yield_groups)
N = sum(len(g) for g in yield_groups)
k = len(yield_groups)
epsilon_sq = (kw.statistic - k + 1) / (N - k)

print(f'Kruskal-Wallis H = {kw.statistic:.3f}, df={k-1}, p = {kw.pvalue:.3g}')
print(f'Effect size (epsilon-squared) = {epsilon_sq:.4f}  -> small effect (Cohen-style bands: <0.01 negligible, 0.01-0.08 small, 0.08-0.26 medium, >0.26 large)')

for s, g in zip(seasons, yield_groups):
    print(f'{s}: median={g.median():.2f}, IQR={g.quantile(.75)-g.quantile(.25):.2f}, n={len(g)}')

Kruskal-Wallis H = 68.604, df=2, p = 1.27e-15
Effect size (epsilon-squared) = 0.0168  -> small effect (Cohen-style bands: <0.01 negligible, 0.01-0.08 small, 0.08-0.26 medium, >0.26 large)
Kharif: median=1.95, IQR=1.92, n=1772
Rabi: median=1.66, IQR=1.73, n=1607
Zaid: median=1.45, IQR=1.44, n=589


In [10]:
def posthoc_mannwhitney(col, group_col='Season', groups=seasons):
    g = {s: df.loc[df[group_col] == s, col].dropna() for s in groups}
    pairs = list(itertools.combinations(groups, 2))
    m = len(pairs)
    rows = []
    for a, b in pairs:
        u = stats.mannwhitneyu(g[a], g[b], alternative='two-sided')
        n1, n2 = len(g[a]), len(g[b])
        rbc = 1 - (2 * u.statistic) / (n1 * n2)
        rows.append({'Group A': a, 'Group B': b, 'U': u.statistic, 'p_raw': u.pvalue,
                     'p_bonferroni': min(u.pvalue * m, 1.0), 'rank_biserial_r': rbc, 'n_A': n1, 'n_B': n2})
    return pd.DataFrame(rows)

yield_posthoc = posthoc_mannwhitney('Yield_Tonnes_Ha')
yield_posthoc

,Group A,Group B,U,p_raw,p_bonferroni,rank_biserial_r,n_A,n_B
0,Kharif,Rabi,"1,568,847.500",0.000,0.000,-0.102,1772,1607
1,Kharif,Zaid,"633,975.500",0.000,0.000,-0.215,1772,589
2,Rabi,Zaid,"528,776.000",0.000,0.000,-0.117,1607,589


In [11]:
df['Cost_Per_Hectare'] = df['Total_Cost_INR'] / df['Farm_Area_Hectares']
cost_groups = [df.loc[df.Season == s, 'Cost_Per_Hectare'].dropna() for s in seasons]

for s, g in zip(seasons, cost_groups):
    print(f'{s}: n={len(g)}, mean={g.mean():,.0f}, median={g.median():,.0f}, skew={stats.skew(g):.3f}')

lev = stats.levene(*cost_groups)
print(f'\nLevene: stat={lev.statistic:.3f}, p={lev.pvalue:.3g}')

kw_cost = stats.kruskal(*cost_groups)
N = sum(len(g) for g in cost_groups); k = len(cost_groups)
eps_cost = (kw_cost.statistic - k + 1) / (N - k)
print(f'Kruskal-Wallis H={kw_cost.statistic:.3f}, p={kw_cost.pvalue:.3g}, epsilon^2={eps_cost:.5f}')

Kharif: n=1779, mean=66,901, median=66,741, skew=-0.064
Rabi: n=1627, mean=66,295, median=66,281, skew=0.074
Zaid: n=594, mean=66,938, median=66,743, skew=0.059

Levene: stat=4.095, p=0.0167
Kruskal-Wallis H=5.022, p=0.0812, epsilon^2=0.00076


In [12]:
d = df.dropna(subset=['Yield_Tonnes_Ha']).copy()
d['rank_yield'] = d['Yield_Tonnes_Ha'].rank()

model = ols('rank_yield ~ C(Season)*C(Crop)', data=d).fit()
aov = sm.stats.anova_lm(model, typ=2)

ms_total_var = np.var(d['rank_yield'], ddof=1)
ss_total = aov['sum_sq'].sum()

srh = aov.copy()
srh['H'] = srh['sum_sq'] / ms_total_var
srh['p_value'] = 1 - stats.chi2.cdf(srh['H'], srh['df'])
srh['eta_sq'] = srh['sum_sq'] / ss_total
srh[['df', 'H', 'p_value', 'eta_sq']]

,df,H,p_value,eta_sq
C(Season),2.000,79.927,0.000,0.020
C(Crop),7.000,"1,864.525",0.000,0.469
C(Season):C(Crop),14.000,8.360,0.870,0.002
Residual,"3,944.000","2,025.510",1.000,0.509


In [13]:
irr_methods = df['Irrigation_Method'].dropna().unique().tolist()
irr_groups = [df.loc[df.Irrigation_Method == m, 'Yield_Tonnes_Ha'].dropna() for m in irr_methods]

kw_irr = stats.kruskal(*irr_groups)
N = sum(len(g) for g in irr_groups); k = len(irr_groups)
eps_irr = (kw_irr.statistic - k + 1) / (N - k)
print(f'Kruskal-Wallis H={kw_irr.statistic:.3f}, p={kw_irr.pvalue:.3g}, epsilon^2={eps_irr:.4f}')

irr_posthoc = posthoc_mannwhitney('Yield_Tonnes_Ha', group_col='Irrigation_Method', groups=irr_methods)
irr_posthoc

Kruskal-Wallis H=22.439, p=5.29e-05, epsilon^2=0.0049


,Group A,Group B,U,p_raw,p_bonferroni,rank_biserial_r,n_A,n_B
0,Drip,Flood,"650,967.000",0.000,0.000,-0.108,907,1296
1,Drip,Rainfed,"520,062.500",0.000,0.000,-0.106,907,1037
2,Drip,Sprinkler,"351,916.000",0.022,0.131,-0.066,907,728
3,Flood,Rainfed,"668,254.000",0.818,1.000,0.006,1296,1037
4,Flood,Sprinkler,"452,552.000",0.128,0.769,0.041,1296,728
5,Rainfed,Sprinkler,"363,259.500",0.178,1.000,0.038,1037,728


In [14]:
num_cols = ['Rainfall_mm','Avg_Temperature_C','Humidity_pct','Sunlight_Hours_Day','Soil_pH',
            'Soil_Moisture_pct','Nitrogen_kg_ha','Phosphorus_kg_ha','Potassium_kg_ha','Fertilizer_kg_ha',
            'Pesticide_Litre_ha','Water_Used_m3','Seed_Quality_Score','Disease_Pest_Risk_pct']

rows = []
for c in num_cols:
    sub = df[[c, 'Yield_Tonnes_Ha']].dropna()
    rs, ps = stats.spearmanr(sub[c], sub['Yield_Tonnes_Ha'])
    rp, pp = stats.pearsonr(sub[c], sub['Yield_Tonnes_Ha'])
    rows.append({'Variable': c, 'Spearman_r': rs, 'Spearman_p': ps, 'Pearson_r': rp, 'Pearson_p': pp})

corr_table = pd.DataFrame(rows).sort_values('Spearman_r', key=abs, ascending=False).reset_index(drop=True)
corr_table

,Variable,Spearman_r,Spearman_p,Pearson_r,Pearson_p
0,Water_Used_m3,0.265,0.000,0.389,0.000
1,Rainfall_mm,0.131,0.000,0.031,0.053
2,Soil_Moisture_pct,0.098,0.000,0.010,0.522
3,Nitrogen_kg_ha,0.082,0.000,0.054,0.001
4,Humidity_pct,0.081,0.000,0.011,0.478
5,Disease_Pest_Risk_pct,0.065,0.000,0.013,0.396
6,Sunlight_Hours_Day,-0.051,0.001,-0.015,0.333
7,Phosphorus_kg_ha,0.048,0.003,0.048,0.003
8,Soil_pH,-0.031,0.050,-0.021,0.193
9,Potassium_kg_ha,0.025,0.113,0.004,0.809


In [15]:
states = df['State'].dropna().unique().tolist()
state_groups = [df.loc[df.State == s, 'Yield_Tonnes_Ha'].dropna() for s in states]
kw_state = stats.kruskal(*state_groups)
N = sum(len(g) for g in state_groups); k = len(state_groups)
eps_state = (kw_state.statistic - k + 1) / (N - k)
print(f'States tested: {len(states)}')
print(f'Kruskal-Wallis H={kw_state.statistic:.3f}, p={kw_state.pvalue:.3g}, epsilon^2={eps_state:.5f}')

States tested: 8
Kruskal-Wallis H=8.605, p=0.282, epsilon^2=0.00041


In [16]:
df['Profit_Per_Hectare'] = df['Profit_INR'] / df['Farm_Area_Hectares']
df['Revenue_Per_Hectare'] = df['Revenue_INR'] / df['Farm_Area_Hectares']
df['Profit_Margin_Pct'] = (df['Profit_INR'] / df['Revenue_INR']) * 100
df['Revenue_Per_Unit_Cost'] = df['Revenue_INR'] / df['Total_Cost_INR']
df['Cost_Per_Tonne_Production'] = df['Total_Cost_INR'] / df['Production_Tonnes']

profit_features = ['Profit_Per_Hectare', 'Profit_Margin_Pct', 'Revenue_Per_Unit_Cost']
results = []
for feat in profit_features:
    g = [df.loc[df.Season == s, feat].dropna() for s in seasons]
    kw = stats.kruskal(*g)
    N = sum(len(x) for x in g); k = len(g)
    eps = (kw.statistic - k + 1) / (N - k)
    medians = {s: gr.median() for s, gr in zip(seasons, g)}
    results.append({'Feature': feat, 'KW_p': kw.pvalue, 'epsilon_sq': eps, **medians})

pd.DataFrame(results)

,Feature,KW_p,epsilon_sq,Kharif,Rabi,Zaid
0,Profit_Per_Hectare,0.000,0.024,"8,340.449",-894.326,"-13,760.259"
1,Profit_Margin_Pct,0.000,0.024,11.090,-1.336,-24.587
2,Revenue_Per_Unit_Cost,0.000,0.024,1.125,0.987,0.803


In [17]:
formula = ('Yield_Tonnes_Ha ~ C(Season) + C(Crop) + C(Irrigation_Method) + Rainfall_mm + '
           'Avg_Temperature_C + Humidity_pct + Sunlight_Hours_Day + Soil_pH + Soil_Moisture_pct + '
           'Nitrogen_kg_ha + Phosphorus_kg_ha + Potassium_kg_ha + Fertilizer_kg_ha + Pesticide_Litre_ha + '
           'Water_Used_m3 + Seed_Quality_Score + Disease_Pest_Risk_pct + Farm_Area_Hectares')
diag_model = ols(formula, data=d).fit()
print(f'R^2 = {diag_model.rsquared:.3f}, Adjusted R^2 = {diag_model.rsquared_adj:.3f}')

R^2 = 0.818, Adjusted R^2 = 0.817
